# Skin Cancer Classifier — Training Notebook

Trains a 7-class skin lesion classifier on **HAM10000** using transfer learning (EfficientNetB0) — round 2, tuned for higher accuracy.
Run on **Kaggle Notebooks with GPU enabled** (right panel → Settings → Accelerator → GPU). You get 30 free GPU hours per week.

**Before you start:**
1. In the notebook editor, click **+ Add Data** (right panel), search `skin-cancer-mnist-ham10000`, and add kmader's dataset.
2. Enable GPU: right panel → Settings → Accelerator → **GPU**.
3. At the end, download `model.h5` + `classes.json` from the Output panel and drop them into the project's `model/` folder.

## 1. Install dependencies

In [ ]:
# Kaggle already ships tensorflow, seaborn and scikit-learn — nothing to install.
print('ready')

## 2. Download HAM10000 from Kaggle

In [ ]:
import os

# The dataset was added via '+ Add Data' in the right panel.
# Find it wherever Kaggle mounted it:
def find_metadata(root='/kaggle/input'):
    for dirpath, _, filenames in os.walk(root):
        if 'HAM10000_metadata.csv' in filenames:
            return os.path.join(dirpath, 'HAM10000_metadata.csv')
    return None

META_PATH = find_metadata()
assert META_PATH, "Dataset not found — click '+ Add Data' and add 'skin-cancer-mnist-ham10000' first."
DATA_DIR = os.path.dirname(META_PATH)
print('dataset at:', DATA_DIR)
print(os.listdir(DATA_DIR))


## 3. Explore the data — check class balance first

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

meta = pd.read_csv(META_PATH)
print(meta.shape)
print(meta['dx'].value_counts())

plt.figure(figsize=(10, 4))
sns.countplot(data=meta, x='dx', order=meta['dx'].value_counts().index)
plt.title('HAM10000 class distribution — note the heavy imbalance')
plt.xticks(rotation=30)
plt.show()

## 4. Map codes to readable names

The backend reads `classes.json` for display names, so keep this mapping — the exported file uses these.

In [ ]:
NAME_MAP = {
    'akiec': 'actinic keratosis',
    'bcc': 'basal cell carcinoma',
    'bkl': 'benign keratosis',
    'df': 'dermatofibroma',
    'mel': 'melanoma',
    'nv': 'nevus',
    'vasc': 'vascular lesion',
}
CLASS_NAMES = [NAME_MAP[c] for c in sorted(NAME_MAP)]  # alphabetical = model output order
print(CLASS_NAMES)

meta['label'] = meta['dx'].map(NAME_MAP)
meta['label_idx'] = meta['dx'].apply(lambda d: sorted(NAME_MAP).index(d))
print(meta[['dx', 'label', 'label_idx']].drop_duplicates().sort_values('label_idx'))

## 5. Stratified train / validation / test split

In [ ]:
import os
from sklearn.model_selection import train_test_split

IMG_DIR_1 = os.path.join(DATA_DIR, 'HAM10000_images_part_1')
IMG_DIR_2 = os.path.join(DATA_DIR, 'HAM10000_images_part_2')

def find_image(image_id):
    for d in (IMG_DIR_1, IMG_DIR_2):
        p = os.path.join(d, image_id + '.jpg')
        if os.path.exists(p):
            return p
    return None

meta['path'] = meta['image_id'].apply(find_image)
meta = meta.dropna(subset=['path']).reset_index(drop=True)
print('images found:', len(meta))

train_df, temp_df = train_test_split(meta, test_size=0.30, random_state=42, stratify=meta['label_idx'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['label_idx'])
print(f'train={len(train_df)}  val={len(val_df)}  test={len(test_df)}')

## 6. Data pipeline — augmentation on train, class weights for imbalance

`nv` (nevus) dominates the dataset. We counter that two ways: **augmentation** (more varied examples of minority classes)
and **class weights** (minority mistakes cost more). This is what prevents the "predicts melanoma for everything" problem.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

IMG_SIZE = (224, 224)
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.25),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

def make_ds(df, training):
    paths = df['path'].values
    labels = df['label_idx'].values.astype(np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    def load(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        if training:
            img = augment(img, training=True)
        img = preprocess_input(tf.cast(img, tf.float32))  # same as backend (efficientnet)!
        return img, label
    ds = ds.map(load, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(2048)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_ds(train_df, training=True)
val_ds = make_ds(val_df, training=False)
test_ds = make_ds(test_df, training=False)

weights = compute_class_weight('balanced', classes=np.unique(train_df['label_idx']), y=train_df['label_idx'])
class_weight = dict(enumerate(weights))
print('class weights:', {CLASS_NAMES[i]: round(w, 2) for i, w in class_weight.items()})

## 7. Model — MobileNetV2 transfer learning

In [ ]:
base = tf.keras.applications.EfficientNetB0(
    weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base.trainable = False  # phase 1: train only the head

model = tf.keras.Sequential([
    base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax'),
])

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

## 8. Train the head

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor='val_loss'),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.3, monitor='val_loss'),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=30,
                    class_weight=class_weight, callbacks=callbacks)

## 9. Fine-tune — unfreeze the top of the base model

In [ ]:
base.trainable = True
for layer in base.layers[:-60]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),  # much lower LR
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history_ft = model.fit(train_ds, validation_data=val_ds, epochs=25,
                       class_weight=class_weight, callbacks=callbacks)

## 10. Evaluate on the held-out test set — per-class, not just overall

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true, y_pred = [], []
for x, y in test_ds:
    y_true.extend(y.numpy())
    y_pred.extend(np.argmax(model.predict(x, verbose=0), axis=1))

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap='Blues')
plt.xlabel('predicted'); plt.ylabel('true'); plt.title('Confusion matrix (test set)')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

acc = np.mean(np.array(y_true) == np.array(y_pred))
print(f'TEST ACCURACY: {acc:.1%}')

## 11. Export — the two files the backend needs

⚠️ **Sanity gate:** only export if every class has reasonable recall in the report above.
If one class dominates the predictions, go back and adjust augmentation / class weights before exporting.

In [ ]:
model.save('/kaggle/working/model.h5')

with open('/kaggle/working/classes.json', 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)

print('exported:', CLASS_NAMES)
print('Done — download model.h5 and classes.json from the Output panel (right sidebar),')
print('put both files in the project model/ folder, and swap in the new backend/model_loader.py.')
